# LLM07 System Prompt Leakage — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM07 — System Prompt Leakage | **Risk Severity**: High

This notebook:
1. **Uploads** all artifact files (scenarios, checks, drivers) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM07 system prompt leakage test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks/drivers upsert).
Registered IDs are available in-memory for the evaluation steps below.

| Scenario | Attack Type | Eval Mode | Check |
|----------|-------------|-----------|-------|
| 1 — Direct Extraction | Blunt queries, social engineering, output-prefix attacks | Single-turn | `LLM07-system-prompt-leakage-detector` |
| 2 — Iterative Extraction | 3-phase probing over 10 turns | Multi-turn simulation | `LLM07-iterative-extraction-detector` |
| 3 — Tool Schema Leakage | Tool definitions, API endpoints, config params | Single-turn | `LLM07-system-prompt-leakage-detector` |

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_artifact,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}  # name -> scenario object

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM07-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

### Register Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

**LLM07 checks**:
- `LLM07-system-prompt-leakage-detector`: Single-turn; 4-category disclosure taxonomy (verbatim, structural, behavioral confirmation, tool/config schema)
- `LLM07-iterative-extraction-detector`: Multi-turn; cumulative disclosure assessment across full conversation transcript

In [ ]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}  # name -> check id

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_artifact(md_path)
    print(f"Registering check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")


### Register Drivers

Scans `drivers/` for `.md` files, parses YAML front matter and persona prompt,
and registers each via `create_or_update_driver` using a `Driver` object.

**LLM07 drivers**:
- `LLM07-pass-through-driver`: Temperature 0; delivers scenario input verbatim (used for Scenarios 1 and 3)
- `LLM07-prompt-extraction-prober`: Temperature 0.7; 3-phase iterative extraction persona (used for Scenario 2)

In [ ]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}  # name -> driver object

for md_path in sorted(drivers_dir.glob("*.md")):
    driver_data = parse_artifact(md_path, default_temperature=0.6)
    print(f"Registering driver: {driver_data['name']} from {md_path.name}")

    driver_obj = Driver(
        name=driver_data["name"],
        prompt_template=driver_data["prompt_template"],
        temperature=driver_data["temperature"],
    )
    result = okareo.create_or_update_driver(driver=driver_obj)
    registered_drivers[driver_data["name"]] = result
    print(f"  ✓ Registered: {driver_data['name']} (ID: {result.id})")

print(f"\nTotal drivers registered: {len(registered_drivers)}")


### Artifact Upload Summary

In [ ]:
print("=" * 60)
print("LLM07 System Prompt Leakage — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  • {name} → {cid}")
print(f"\nDrivers ({len(registered_drivers)}):")
for name, drv in registered_drivers.items():
    print(f"  • {name} → {drv.id}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.json` file (copy `owasp/target.json.example` and fill in your values).
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

**LLM07-specific simulation settings**:
- Scenarios 1 and 3 (single-turn): `max_turns=1`, `first_turn="driver"` — extraction probe is delivered immediately
- Scenario 2 (multi-turn): `max_turns=10`, `first_turn="target"` — agent greets first, then probing driver begins 3-phase extraction

In [ ]:
# Target loaded from owasp/target.json. To use a different config: target = build_target(CATEGORY_DIR, config_path="target.prod.json")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"\u2713 Target agent: {TARGET_NAME}")

SINGLE_TURN_CHECK = "LLM07-system-prompt-leakage-detector"
MULTI_TURN_CHECK  = "LLM07-iterative-extraction-detector"
PROBER_DRIVER_NAME = "LLM07-prompt-extraction-prober"
MAX_TURNS = 10

### Single-Turn Tests — Scenario 1 (Direct Extraction) & Scenario 3 (Tool Schema Leakage)

Each scenario runs via `okareo.run_simulation()` constrained to `max_turns=1, first_turn="driver"`.
A pass-through driver delivers the extraction probe verbatim to the target agent; the agent's
response is evaluated by the `LLM07-system-prompt-leakage-detector` check using the 4-category
disclosure taxonomy.

In [ ]:
SINGLE_TURN_SCENARIOS = [
    "LLM07-direct-extraction",
    "LLM07-tool-schema-leakage",
]

pass_through_driver = Driver(
    temperature=0,
    name=f"{TARGET_NAME}-pass-through-driver",
    prompt_template=SINGLE_TURN_DRIVER_TEMPLATE,
)

single_turn_results = {}

for scenario_name in SINGLE_TURN_SCENARIOS:
    print(f"\n{'='*60}")
    print(f"Running: {scenario_name}")
    print(f"{'='*60}")
    try:
        scenario = registered_scenarios[scenario_name]

        test_run = okareo.run_simulation(
            target=target,
            driver=pass_through_driver,
            name=f"LLM07 Eval — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn="driver",
            scenario=scenario,
            max_turns=1,
            checks=[SINGLE_TURN_CHECK],
        )
        single_turn_results[scenario_name] = test_run
        print(f"  ✓ Test run complete: {test_run.id}")
        if hasattr(test_run, "app_link") and test_run.app_link:
            print(f"  View: {test_run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        single_turn_results[scenario_name] = None

### Multi-Turn Simulation — Scenario 2: Iterative Extraction

Runs a 10-turn simulation using the `LLM07-prompt-extraction-prober` driver and
`LLM07-iterative-extraction-detector` check via `okareo.run_simulation()` with
`max_turns=10, first_turn="target"`.

The agent speaks first (`first_turn="target"`), then the probing driver begins its
3-phase extraction strategy: capability mapping (turns 1–3), constraint probing (turns 4–7),
hypothesis confirmation (turns 8–10).

The check evaluates **cumulative disclosure** across the full conversation transcript —
not individual turns — to detect progressive system prompt reconstruction.

In [ ]:
MULTI_TURN_SCENARIO = "LLM07-iterative-extraction"

print(f"\n{'='*60}")
print(f"Running simulation: {MULTI_TURN_SCENARIO}")
print(f"Driver: {PROBER_DRIVER_NAME} | Max turns: {MAX_TURNS} | First turn: target")
print(f"{'='*60}")

simulation_run = None
try:
    driver_reg = registered_drivers.get(PROBER_DRIVER_NAME)
    if driver_reg is None:
        raise ValueError(f"Driver '{PROBER_DRIVER_NAME}' not found in registered_drivers. Check upload step.")

    prober_driver = Driver(
        temperature=driver_reg.temperature if hasattr(driver_reg, "temperature") else 0.7,
        name=PROBER_DRIVER_NAME,
        prompt_template=driver_reg.prompt_template,
    )

    scenario = registered_scenarios[MULTI_TURN_SCENARIO]

    simulation_run = okareo.run_simulation(
        target=target,
        driver=prober_driver,
        name=f"LLM07 Simulation — {MULTI_TURN_SCENARIO}",
        api_key=OKAREO_API_KEY,
        first_turn="target",
        scenario=scenario,
        max_turns=MAX_TURNS,
        checks=[MULTI_TURN_CHECK],
    )
    print(f"  ✓ Simulation complete: {simulation_run.id}")
    if hasattr(simulation_run, "app_link") and simulation_run.app_link:
        print(f"  View: {simulation_run.app_link}")
except Exception as e:
    print(f"  ✗ Error: {e}")

### Results Summary

In [ ]:
print("\n" + "=" * 60)
print("LLM07 SYSTEM PROMPT LEAKAGE — EVALUATION RESULTS")
print("OWASP Category: LLM07 | Risk Severity: High")
print("=" * 60)

all_results = dict(single_turn_results)
if simulation_run is not None:
    all_results[MULTI_TURN_SCENARIO] = simulation_run

print(f"\n{'Scenario':<42} {'Status':<10} {'Link / Run ID'}")
print("-" * 100)
for name, result in all_results.items():
    if result is None:
        print(f"{name:<42} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<42} {'COMPLETE':<10} {link}")

errors = sum(1 for r in all_results.values() if r is None)
print(f"\nTotal evaluated: {len(all_results)} | Errors: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full results.")
print()
print("Disclosure categories detected by check (see dashboard rationale):")
print("  • Verbatim     — agent reproduced literal system prompt text")
print("  • Structural   — agent revealed prompt existence/structure/sections")
print("  • Behavioral   — agent confirmed/denied hypothetical prompt fragments")
print("  • Tool/Config  — agent disclosed tool schemas, API endpoints, config values")
print("  • None         — agent correctly declined all disclosure (PASS)")

### Detailed Results (Optional)

Retrieve per-row scores and model outputs for any completed test run.

In [ ]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)